# Sampler Validation: Marginal Checks

For each validation target, we run the Boomerang,resample uniformly in time, 
and overlay sample histograms against the known marginal densities.

In [ ]:
import os
os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt

from benchmarks_august.targets.validation import gaussian, gaussian_mixture, banana
from benchmarks_august.samplers.factories import build_sampler
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.samplers.warmstart import warmup_reference

In [ ]:
# ── Shared settings ──────────────────────────────────────────────
N_SKELETON  = 300000
N_RESAMPLE  = 300000
BURNIN_FRAC = 0.1
refresh_rate = 1.0

def run_and_resample(sampler, target, sticky=False, warmup=True, N_SKELETONS=N_SKELETON):
    """Warmup, preprocess, sample, and return time-uniform resamples."""
    if warmup:
        warmup_reference(sampler, n_rounds=3, n_pilot=500,
                         sticky=sticky, target=target)
    else:
        method = target.meta.get('preprocess_method', 'diagonal')
        if method == 'manual':
            sampler.preprocess(method='manual',
                               x_ref=target.x_ref,
                               Sigma_inv=target.Sigma_inv)
        else:
            sampler.preprocess(method='diagonal')
    
    sampler.reset(N=N_SKELETONS)
    sampler.sample_auto(diagnostics=True)
    
    if sticky:
        _, samples = resample_sticky_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                               burnin_frac=BURNIN_FRAC)
    else:
        _, samples = resample_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                        burnin_frac=BURNIN_FRAC)
    return samples


def plot_marginals(target, samples_dict, figname=None, bins=50):
    """Plot marginal histograms against true densities for each coordinate."""
    marginals = target.meta['marginal_grids']
    D = target.D
    n_samplers = len(samples_dict)

    fig, axes = plt.subplots(n_samplers, D, figsize=(3.5 * D, 3 * n_samplers),
                             squeeze=False)

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']

    for row, (label, samples) in enumerate(samples_dict.items()):
        for col in range(D):
            ax = axes[row, col]
            mg = marginals[col]
            grid, pdf = mg['grid'], mg['pdf']

            x = samples[:, col]
            # Clip histogram range to the reference grid so the overlay aligns
            x_lo, x_hi = float(grid[0]), float(grid[-1])
            ax.hist(x, bins=bins, range=(x_lo, x_hi), density=True, alpha=0.5,
                    color=colors[row % len(colors)],
                    label=label if (row == 0 and col == 0) else None)
            ax.plot(grid, pdf, 'k-', lw=1.5,
                    label='True' if (row == 0 and col == 0) else None)
            ax.set_xlim(x_lo, x_hi)

            # Annotate out-of-range sample mass if nontrivial
            frac_out = np.mean((x < x_lo) | (x > x_hi))
            if frac_out > 0.01:
                ax.text(0.02, 0.95, f"{frac_out:.1%} out of range",
                        transform=ax.transAxes, fontsize=7,
                        va='top', color='firebrick')

            if row == 0:
                ax.set_title(mg['label'])
            if col == 0:
                ax.set_ylabel(label)
            if row == n_samplers - 1:
                ax.set_xlabel(mg['label'])

    # One legend for the whole figure
    handles, labels_ = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, loc='upper right',
                   bbox_to_anchor=(0.99, 0.99), fontsize=9)

    fig.suptitle(target.name, fontsize=14, y=1.02)
    fig.tight_layout()
    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight')
    plt.show()  
    

## 1. Gaussian sanity check

In [ ]:
diag_gauss = gaussian(D=5)
ar_gauss = gaussian(D=5, cov="ar1")
random_gauss = gaussian(D=5, cov="random")

target_gauss = random_gauss

s_gauss = build_sampler('boomerang', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate)
samp_gauss = run_and_resample(s_gauss, target_gauss, warmup=False)

s_gauss_pli = build_sampler('boomerang_pli', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate)
samp_gauss_pli = run_and_resample(s_gauss_pli, target_gauss, warmup=False)

# Check: should be zero bounces
df = s_gauss.diagnostics_df
n_bounces = df[(df['event_type'] == 'bounce') & (df['accepted'] == True)].shape[0]
n_refresh = df[df['event_type'] == 'refresh'].shape[0]
wall = df['wall_seconds'].sum()
grad_evals = s_gauss.gradient_evals
df_pli = s_gauss_pli.diagnostics_df
n_bounces_pli = df_pli[(df_pli['event_type'] == 'bounce') & (df_pli['accepted'] == True)].shape[0]
n_refresh_pli = df_pli[df_pli['event_type'] == 'refresh'].shape[0]
wall_pli = df_pli['wall_seconds'].sum()
grad_evals_pli = s_gauss_pli.gradient_evals

print("--------- Boomerang ---------")
print(f"Accepted bounces: {n_bounces}  (expect 0)")
print(f"Refreshments:     {n_refresh}  (expect all skeleton points)")
print(f"Walltime:         {wall}")
print(f"Grad evals per skeleton point: {grad_evals / s_gauss.N:.1f}")
print("--------- Boomerang PLI ---------")
print(f"Accepted bounces: {n_bounces_pli}  (expect 0)")
print(f"Refreshments:     {n_refresh_pli}  (expect all skeleton points)")
print(f"Walltime:         {wall_pli}")
print(f"Grad evals per skeleton point: {grad_evals_pli / s_gauss_pli.N:.1f}")

plot_marginals(target_gauss, {'Boomerang': samp_gauss,
                              'Boomerang PLI': samp_gauss_pli})
               # ,figname='validation_gaussian_refcheck.pdf')

## 2. Gaussian Mixture

In [ ]:
bimodal_mixture = gaussian_mixture(D=1, preset="bimodal")
heavy_tails_mixture = gaussian_mixture(D=1, preset="heavy_tailed")
skewed_mixture = gaussian_mixture(D=1, preset="skewed")


target_mixture = bimodal_mixture
#target_mixture.x_ref = np.array([0.0, 0.0])
target_mixture.x_ref = np.array([0.0])
#target_mixture.Sigma_inv = 0.1*np.eye(2)
target_mixture.Sigma_inv = 0.1*np.eye(1)
#target_mixture.Sigma_inv = np.eye(1)
target_mixture.meta['preprocess_method'] = 'manual'

**Reference specification.** For a multimodal target, automatic warm-starting fits a unimodal Gaussian to one basin, and the sampler then orbits that mode without visiting the others. We therefore specify the reference manually: $x_{\mathrm{ref}} = 0$ placed between the modes, and $\Sigma = 10 \cdot I$ chosen so the reference ellipses geometrically cover both modes.

**Refreshment rate.** The Boomerang's deterministic flow on a Gaussian reference $\mathcal{N}(x_{\mathrm{ref}}, \Sigma)$ is periodic with period $2\pi$ in trajectory time. The refreshment rate $\lambda_{\mathrm{ref}}$ sets the mean free-flight time $1/\lambda_{\mathrm{ref}}$; we want this to be of the order of one half-orbit so the trajectory can traverse the target support between velocity redraws (in free flight). This gives the default

$$
\lambda_{\mathrm{ref}} \;\approx\; \frac{1}{\pi} \;\approx\; 0.3.
$$

When the reference is deliberately mismatched to the target, the rate scales with the ratio of reference to target length scales,

$$
\lambda_{\mathrm{ref}} \;\approx\; \frac{1}{\pi} \cdot \frac{\sigma_{\mathrm{ref}}}{L},
$$

where $L$ is the characteristic target extent (mode separation) and $\sigma_{\mathrm{ref}} = \sqrt{\max_i \Sigma_{ii}}$. For the bimodal mixture, $L = 6$ and $\sigma_{\mathrm{ref}} = \sqrt{10} \approx 3.2$, giving $\lambda_{\mathrm{ref}} \approx 0.15$.

In [ ]:
# Boomerang
s_bb = build_sampler('boomerang', target_mixture, N=N_SKELETON, refresh_rate=0.1)
samp_bb = run_and_resample(s_bb, target_mixture, warmup=False)

In [ ]:
# Boomerang PLI
s_bb_pli = build_sampler('boomerang_pli', target_mixture, N=N_SKELETON, refresh_rate=0.1)
samp_bb_pli = run_and_resample(s_bb_pli, target_mixture, warmup=False)

In [ ]:
# Boomerang PLI
s_bb_pli_refreshed = build_sampler('boomerang_pli', target_mixture, N=N_SKELETON, refresh_rate=10.0)
samp_bb_pli_refreshed = run_and_resample(s_bb_pli_refreshed, target_mixture, warmup=False)

In [ ]:
plot_marginals(target_mixture, {
    #'Boomerang': samp_bb,
    'Boomerang PLI': samp_bb_pli,
    'Boomerang PLI refreshed': samp_bb_pli_refreshed,
})#, figname='validation_beta_binomial.pdf')


In [ ]:
# print("------Boomerang------")
# print("x_ref:", s_bb.x_ref)
# print("diag(Sigma):", np.diag(np.linalg.inv(s_bb.Sigma_inv)))
# print("sample mean per coord:", samp_bb.mean(axis=0))
# print("sample std per coord:", samp_bb.std(axis=0))
# # Fraction of samples near each mode
# near_neg = np.mean(samp_bb[:, 0] < -1)
# near_pos = np.mean(samp_bb[:, 0] >  1)
# print(f"Mass below -1: {near_neg:.3f}, above +1: {near_pos:.3f}, (expect ~0.5 each)")

print("------Boomerang PLI------")
print("x_ref:", s_bb_pli.x_ref)
print("diag(Sigma):", np.diag(np.linalg.inv(s_bb_pli.Sigma_inv)))
print("sample mean per coord:", samp_bb_pli.mean(axis=0))
print("sample std per coord:", samp_bb_pli.std(axis=0))
# Fraction of samples near each mode
near_neg = np.mean(samp_bb_pli[:, 0] < -1)
near_pos = np.mean(samp_bb_pli[:, 0] >  1)
print(f"Mass below -1: {near_neg:.3f}, above +1: {near_pos:.3f}, (expect ~0.5 each)")

In [ ]:
print("------Boomerang------")
df = s_bb.diagnostics_df
n_bounce_accepted = ((df['event_type'] == 'bounce') & (df['accepted'])).sum()
n_refresh = (df['event_type'] == 'refresh').sum()
print(f"Accepted: {n_bounce_accepted}")
print(f"Refreshes: {n_refresh}")
print(f"Sum: {n_bounce_accepted + n_refresh}")
print(f"Internal counter: {df['event_type'].value_counts()}")
print("------Boomerang PLI------")
df = s_bb_pli.diagnostics_df
n_bounce_accepted = ((df['event_type'] == 'bounce') & (df['accepted'])).sum()
n_refresh = (df['event_type'] == 'refresh').sum()
print(f"Accepted: {n_bounce_accepted}")
print(f"Refreshes: {n_refresh}")
print(f"Sum: {n_bounce_accepted + n_refresh}")
print(f"Internal counter: {df['event_type'].value_counts()}")

### Notes

**PLI degradation at low refreshment rates.** The PLI variant approximates the switching rate along each proposal segment by a linear interpolant $h(t) = a_i t + b_i$ fitted through two rate evaluations at the segment endpoints. For Gaussian targets the true rate is of the form $A + B\cos(2t) + C\sin(2t)$, which is extremely well-approximated by a chord over short segments, so on smooth, near-quadratic targets the envelope is effectively a true upper bound. When the target is strongly non-Gaussian along the trajectory (for example when a trajectory segment sweeps across the valley of a multimodal target), the rate has higher-frequency structure that the chord cannot capture, and the "bound" is violated. Short segments, induced by a high $\lambda_{\mathrm{ref}}$, keep the trajectory locally Gaussian-like and the envelope holds. As $\lambda_{\mathrm{ref}}$ decreases, segments grow and the envelope fails. On this target we observe a maximum true-rate-to-envelope ratio of $\sim 2.6$ at $\lambda_{\mathrm{ref}} = 10$ versus $\sim 76$ at $\lambda_{\mathrm{ref}} = 0.1$, and more bound violations during sampling. Missed events cause the PLI samples to drift toward the reference measure, failing to capture the multimodality.

**Possible fixes.** Two aspects of the current PLI implementation could be revised. First, the initial subinterval $t_{\text{init}} = \text{horizon}/100$ is chosen non-adaptively. Since the linear envelope is exact for Gaussian targets and fails on non-Gaussian ones, an adaptive choice that evaluates the rate at a midpoint $t_{\text{init}}/2$ and checks consistency with the chord would detect non-Gaussianity in the segment and shorten $t_{\text{init}}$ as needed, at the cost of one extra rate evaluation per segment and possible more interpolations. Second, when the ratio between the true rate and the envelope exceeds 2, Goan et al. reject the proposed time and resample. Our implementation instead advances time by the current horizon. Adopting the rejection rule recovers the correct thinning semantics in the violation regime.

In [ ]:
from Filippo_plotting.mcmc_plots import better_pairs

labels_bb = [target_gauss.meta['marginal_grids'][i]['label'] for i in range(target_gauss.D)]

fig, axes = better_pairs(samp_bb, resol=0.7, labels=labels_bb,
                         title='Boomerang beta-binomial')
plt.show()

fig, axes = better_pairs(samp_bb_pli, resol=0.7, labels=labels_bb,
                         title='Boomerang PLI beta-binomial')
plt.show()

## 3. Rosenbrocks banana

In [ ]:
simple_banana = banana(D=2, a=1.0, scale=1.0)
large_banana = banana(D=10, a=1.0, scale=1.0)
anisotropic_banana = banana(D=2, a=1.0, scale=10.0)

target_banana = simple_banana
target_banana.x_ref = np.array([0.0, 1.0])
target_banana.Sigma_inv = np.diag([1, 3])
target_banana.meta['preprocess_method'] = 'manual'

### Preprocessing the banana

The banana has no affine match to a Gaussian, so no choice of $(x_{\mathrm{ref}}, \Sigma)$ tracks its curvature — we can only match moments. The exact $\beta_0$ marginal is $\mathcal{N}(0, 1)$ and the $\beta_1$ marginal has mean $1$ and variance $\approx 3$, so we set
$$
x_{\mathrm{ref}} = (0, 1), \qquad \Sigma = \mathrm{diag}(1, 3).
$$
The trajectories cut across the arms rather than following them, but sampling remains correct.

For the refreshment rate we apply the same heuristic as above,
$$
\lambda_{\mathrm{ref}} \;\approx\; \frac{1}{\pi} \cdot \frac{\sigma_{\mathrm{ref}}}{L},
$$
with $L$ the arm length from mode to tip ($\approx 5$) and $\sigma_{\mathrm{ref}} = \sqrt{3} \approx 1.7$, giving $\lambda_{\mathrm{ref}} \approx 0.1$.

In [ ]:
# Boomerang
s_banana = build_sampler('boomerang', target_banana, N=N_SKELETON, refresh_rate=0.1)
samp_banana = run_and_resample(s_banana, target_banana, warmup=False, N_SKELETONS=N_SKELETON)

In [ ]:
# Boomerang PLI
s_banana_pli = build_sampler('boomerang_pli', target_banana, N=N_SKELETON, refresh_rate=0.1)
samp_banana_pli = run_and_resample(s_banana_pli, target_banana, warmup=False, N_SKELETONS=N_SKELETON)

In [ ]:
# Boomerang PLI
s_banana_pli_refresh = build_sampler('boomerang_pli', target_banana, N=N_SKELETON, refresh_rate=10.0)
samp_banana_pli_refresh = run_and_resample(s_banana_pli_refresh, target_banana, warmup=False, N_SKELETONS=N_SKELETON)

In [ ]:
plot_marginals(target_banana, {
    #'Boomerang': samp_banana,
    'Boomerang PLI': samp_banana_pli,
    'Boomerang PLI refreshed': samp_banana_pli_refresh,
})#, figname='validation_neals_funnel.pdf')

In [ ]:
print("------Boomerang------")
print("x_ref:", s_banana.x_ref)
print("diag(Sigma):", np.diag(np.linalg.inv(s_banana.Sigma_inv)))
print("sample mean per coord:", samp_banana.mean(axis=0))
print("sample std per coord:", samp_banana.std(axis=0))

# print("------Boomerang PLI------")
print("x_ref:", s_banana_pli.x_ref)
print("diag(Sigma):", np.diag(np.linalg.inv(s_banana_pli.Sigma_inv)))
print("sample mean per coord:", samp_banana_pli.mean(axis=0))
print("sample std per coord:", samp_banana_pli.std(axis=0))

In [ ]:
print("------Boomerang------")
df = s_banana.diagnostics_df
n_bounce_accepted = ((df['event_type'] == 'bounce') & (df['accepted'])).sum()
n_refresh = (df['event_type'] == 'refresh').sum()
print(f"Accepted: {n_bounce_accepted}")
print(f"Refreshes: {n_refresh}")
print(f"Sum: {n_bounce_accepted + n_refresh}")
print(f"Internal counter: {df['event_type'].value_counts()}")
print("------Boomerang PLI------")
df = s_banana_pli.diagnostics_df
n_bounce_accepted = ((df['event_type'] == 'bounce') & (df['accepted'])).sum()
n_refresh = (df['event_type'] == 'refresh').sum()
print(f"Accepted: {n_bounce_accepted}")
print(f"Refreshes: {n_refresh}")
print(f"Sum: {n_bounce_accepted + n_refresh}")
print(f"Internal counter: {df['event_type'].value_counts()}")

### Notes

We observe roughly the same behaviour as for the mixture, with perhaps worse results.

## INVESTIGATION OF THE RATES

In [ ]:
from functools import partial
from scipy.linalg import cholesky
import autograd.numpy as anp

def gradU(x, Sigma_inv, grad_target, x_ref):
    """
    Gradient accounting for reference measure
    """
    return grad_target(x) - Sigma_inv@(x-x_ref)

def trajectory(t, x, v, x_ref):
    """
    Boomerang trajectory
    """
    x_t = x_ref + (x - x_ref) * anp.cos(t) + v * anp.sin(t)
    v_t = -(x - x_ref) * anp.sin(t) + v * anp.cos(t)
    return x_t, v_t

def rate(t, x, v, x_ref, Sigma_inv, target):
    """
    Boomerang negative rate
    """
    x_t, v_t = trajectory(t, x, v, x_ref)
    inner = anp.dot(v_t, gradU(x_t, Sigma_inv=Sigma_inv, grad_target=target.gradE, x_ref=x_ref))
    return inner

# x_ref = np.array([0.0, 1.0])
# Sigma_inv = np.diag([1])
x_ref = np.array([0.0, 1.0])
Sigma_inv = np.diag([1, 3])
Sigma = np.linalg.inv(Sigma_inv)
Sigma = 0.5 * (Sigma + Sigma.T)
Sigma_sqrt = cholesky(Sigma, lower=True)

pos = x_ref + Sigma_sqrt @ np.random.randn(2)
vel = Sigma_sqrt @ np.random.randn(2)

In [ ]:
# ── Rate comparison across targets ────────────────────────────────

# Horizon long enough to show multiple oscillations
t_grid = np.linspace(0, 10, 1000)

# Build one target per family
targets = {
    'Gaussian':  gaussian(D=2),
    'Mixture':   gaussian_mixture(D=2, preset='bimodal'),
    'Banana':    banana(D=2, a=1.0, scale=1.0),
}

# Reference (x_ref, Sigma) per target — matched to what each sampler would use
refs = {
    'Gaussian':  (np.array([0.0, 0.0]), np.eye(2)),           # matched reference
    'Mixture':   (np.array([0.0, 0.0]), 10.0 * np.eye(2)),    # overdispersed
    'Banana':    (np.array([0.0, 1.0]), np.diag([1.0, 3.0])), # moment-matched
}


In [ ]:
# Fix a single (x_0, v_0) per target so the curves are directly comparable
# in shape (same random direction, same scale relative to the reference).
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

for ax, (name, target) in zip(axes, targets.items()):
    x_ref, Sigma = refs[name]
    Sigma_inv = np.linalg.inv(Sigma)
    Sigma_sqrt = cholesky(Sigma, lower=True)

    # initial position drawn from reference, velocity from N(0, Sigma^{-1})
    x0 = x_ref + Sigma_sqrt @ rng.standard_normal(2)
    v0 = np.linalg.solve(Sigma_sqrt.T, rng.standard_normal(2))

    rate_t = partial(rate, x=x0, v=v0, x_ref=x_ref,
                     Sigma_inv=Sigma_inv, target=target)
    rates     = np.array([rate_t(t) for t in t_grid])
    rates_pos = np.maximum(rates, 0.0)

    ax.plot(t_grid, rates,     color='0.5', lw=1.0, label=r'$\langle v, \nabla U\rangle$')
    ax.plot(t_grid, rates_pos, color='black', lw=1.8, label=r'$\lambda(t)$')
    ax.axhline(0.0, color='red', ls='--', lw=0.8)
    ax.set_title(name)
    ax.set_xlabel('trajectory time $t$')
    if ax is axes[0]:
        ax.set_ylabel('rate')
    ax.legend(loc='upper right', fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
# Sample rate over N periods to resolve integer frequencies cleanly
N_PERIODS = 40        # 40 * 2*pi seconds
N_SAMPLES = 4000
t_grid = np.linspace(0.0, 2*np.pi*N_PERIODS, N_SAMPLES, endpoint=False)
dt = t_grid[1] - t_grid[0]

rng = np.random.default_rng(0)
z_x = rng.standard_normal(2)
z_v = rng.standard_normal(2)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col, (name, target) in enumerate(targets.items()):
    x_ref, Sigma = refs[name]
    Sigma_inv = np.linalg.inv(Sigma)
    Sigma_sqrt = cholesky(Sigma, lower=True)
    x0 = x_ref + Sigma_sqrt @ z_x
    v0 = np.linalg.solve(Sigma_sqrt.T, z_v)

    rates = np.array([rate(t, x0, v0, x_ref, Sigma_inv, target) for t in t_grid])

    # FFT: since the rate is periodic with period 2*pi, integer frequencies
    # (in units of 1 cycle per 2*pi) correspond to FFT bins k = N_PERIODS, 2*N_PERIODS, ...
    F = np.abs(np.fft.rfft(rates))
    freqs_bins = np.fft.rfftfreq(len(rates), d=dt) * 2*np.pi   # now in "mode k" units

    # Plot top row: rate on first 10 trajectory-time units
    ax_top = axes[0, col]
    mask = t_grid <= 10.0
    ax_top.plot(t_grid[mask], rates[mask], color='black', lw=1.0)
    ax_top.axhline(0, color='red', ls='--', lw=0.6)
    ax_top.set_title(f'{name}: rate on t ∈ [0, 10]')
    ax_top.set_xlabel('t')

    # Plot bottom row: FFT amplitude vs mode number
    ax_bot = axes[1, col]
    max_mode = 8
    ax_bot.stem(freqs_bins[freqs_bins <= max_mode],
                F[freqs_bins <= max_mode], basefmt=' ')
    ax_bot.set_xlabel('frequency mode k (cycles per 2π)')
    ax_bot.set_ylabel('|FFT|')
    ax_bot.set_title(f'{name} spectrum')
    # annotate which modes are nonzero
    top_modes = []
    for k in range(1, max_mode+1):
        idx = np.argmin(np.abs(freqs_bins - k))
        amp = F[idx]
        top_modes.append((k, amp))
    # report
    print(f"\n--- {name} ---")
    total = sum(a for _, a in top_modes)
    for k, amp in top_modes:
        frac = amp / total if total > 0 else 0
        bar = '#' * int(frac * 40)
        print(f"  mode {k}: amp={amp:8.2f}  {bar}")

In [ ]:
t_grid = np.linspace(0, 10, 1000)

diag_gauss = gaussian(D=2)
target_gauss = diag_gauss

bimodal_mixture = gaussian_mixture(D=2, preset="bimodal")
target_mixture = bimodal_mixture

simple_banana = banana(D=2, a=1.0, scale=1.0)
target_banana = simple_banana

rate_time = partial(rate, x=pos, v=vel, x_ref=x_ref, Sigma_inv=Sigma_inv, target = simple_banana)

rate_time_vec = anp.vectorize(rate_time)
rates = anp.array([rate_time(t) for t in t_grid])

plt.figure()
plt.plot(t_grid, rates)
plt.hlines(0, xmin=t_grid[0], xmax=t_grid[-1], linestyles="dashed", color="red")
plt.show()

**Frequency content of the rate.** For a Gaussian target with matched
reference, the rate is a pure mode-2 oscillation $\lambda(t) = A +
B\cos(2t) + C\sin(2t)$. For a bimodal mixture with overdispersed
reference, within a single basin the rate is dominated by mode 2 with
a weaker mode-4 component (and higher even harmonics). For the banana
with moment-matched reference, the rate contains modes 1, 2, 3, and 4
with comparable amplitudes — confirmed empirically by FFT of the rate
function along a trajectory. A linear chord over $[0, T]$ represents
only modes 0 and 1 exactly; all higher modes contribute to the
chord-vs-rate deviation. For the banana, with modes up to 4 present,
the chord fails when $T$ becomes comparable to a substantial fraction
of the shortest present period, $2\pi/4 = \pi/2 \approx 1.57$. Observed
mean bounce horizons of 11.8 in the PLI diagnostics correspond to
~7.5 periods of the mode-4 component — far beyond where the chord
approximation is valid.